Retrieves document better than semantic search from vector stores,
uses strategy (semantic search, MMR, scoring factor)

# Vector Store Retriever

In [1]:
2+2

4

## using vector store - not accurate

In [ ]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings 
from langchain_community.vectorstores import FAISS

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)


# Create retriever
# retriever = vs.as_retriever(search_kwargs={"k": 3})

# Query
query = "Which animals are domestic pets?"
# out_docs = retriever.invoke(query)
out_docs = vs.similarity_search("query", k=3)

print("Retriever results:")
for d in out_docs:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

Retriever results:
ID: 1, Content: Python is a programming language
ID: 2, Content: Java is also a programming language
ID: 5, Content: Birds fly in the sky


## using retriever - accurate

In [ ]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings  
from langchain_community.vectorstores import FAISS

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)

# Create retriever
# default ssearch type = semantic search, k=3
retriever = vs.as_retriever(search_kwargs={"k": 3})

# Query
query = "Which animals are domestic pets?"
out_docs = retriever.invoke(query)

print("Retriever results:")
for d in out_docs:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

c:\Users\GAYATHRI DEVI\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retriever results:
ID: 4, Content: Dogs bark at strangers
ID: 5, Content: Birds fly in the sky
ID: 3, Content: Cats sleep often during the day


In [4]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings 
from langchain_community.vectorstores import FAISS

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)

# Create retriever
retriever = vs.as_retriever(search_type = "mmr", search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5})

# Query
query = "Which animals are domestic pets?"
out_docs = retriever.invoke(query)

print("Retriever results:")
for d in out_docs:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

Retriever results:
ID: 4, Content: Dogs bark at strangers
ID: 1, Content: Python is a programming language
ID: 3, Content: Cats sleep often during the day


# BM25 Retriever

In [5]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Create 5 Document objects
docs = [
    Document(page_content="The sky is blue during the day", metadata={"id": 1}), # sky, blue
    Document(page_content="At night the stars light up the sky", metadata={"id": 2}), # sky
    Document(page_content="Blue whales are the largest animals on Earth", metadata={"id": 3}), # blue animal
    Document(page_content="Birds can fly high up in the sky", metadata={"id": 4}), # sky
    Document(page_content="Deep sea creatures live in darkness", metadata={"id": 5}), # No matching
]

# Build BM25 retriever
bm25 = BM25Retriever.from_documents(docs, k=3)

# Run query
results = bm25.invoke("sky blue animals")

print("Top 3 results:")
for d in results:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

Top 3 results:
ID: 1, Content: The sky is blue during the day
ID: 3, Content: Blue whales are the largest animals on Earth
ID: 4, Content: Birds can fly high up in the sky


# Ensemble / Hybrid Retriever

In [ ]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
# from langchain.retrievers import EnsembleRetriever
from langchain_classic.retrievers import EnsembleRetriever

# 5 documents
docs = [
    Document(page_content="AI research is growing fast", metadata={"id": 1}),
    Document(page_content="Machine learning models perform many tasks", metadata={"id": 2}),
    Document(page_content="Cooking and baking are arts", metadata={"id": 3}),
    Document(page_content="Dogs are loyal animals", metadata={"id": 4}),
    Document(page_content="Software engineering is a discipline", metadata={"id": 5}),
]

# BM25 retriever
bm25 = BM25Retriever.from_documents(docs, k=2)

# Vector retriever
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vs = FAISS.from_documents(docs, embedding=emb)
vector_retriever = vs.as_retriever(search_kwargs={"k": 2})

# Ensemble
ensemble = EnsembleRetriever(retrievers=[bm25, vector_retriever], weights=[0.5, 0.5])

# Query
query = "what are models in AI"
results = ensemble.invoke(query)
print("Ensemble Results:")
for d in results:
    print(f"ID {d.metadata.get('id')}: {d.page_content}")

c:\Users\GAYATHRI DEVI\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ensemble Results:
ID 1: AI research is growing fast
ID 2: Machine learning models perform many tasks


# Contextual Compression Retriever

In [3]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

# 5 Document objects
docs = [
    Document(page_content="Python is a programming language used for web dev and data science", metadata={"id":1}),
    Document(page_content="JavaScript runs in the browser, useful for interactive web pages", metadata={"id":2}),
    Document(page_content="C++ is compiled and used for system programming and games", metadata={"id":3}),
    Document(page_content="Rust aims for safety and performance, used in systems software", metadata={"id":4}),
    Document(page_content="Cooking recipes often require precise measurement and timing", metadata={"id":5}),
]

# Base retriever: vector store
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vs = FAISS.from_documents(docs, embedding=emb)
base_retriever = vs.as_retriever(search_kwargs={"k":3})

# Compressor: drop docs whose embedding is too dissimilar
compressor = EmbeddingsFilter(embeddings=emb, similarity_threshold=0.5)

# ContextualCompressionRetriever
cc_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

# Query and full output
query = "Which languages are good for system programming?"
out_docs = cc_retriever.invoke(query)

print("Contextual Compression Results:")
for d in out_docs:
    print(f"ID {d.metadata.get('id')}: {d.page_content}")

Contextual Compression Results:
ID 3: C++ is compiled and used for system programming and games
ID 4: Rust aims for safety and performance, used in systems software
ID 1: Python is a programming language used for web dev and data science


# MultiQuery Retriever

## generally creates 3 extra queries +1 original query

In [ ]:
from langchain_ollama import ChatOllama 
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

import logging

logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)


# 5 Document objects
docs = [
    Document(page_content="Machine learning includes supervised, unsupervised, and reinforcement learning", metadata={"id":1}),
    Document(page_content="Deep learning uses neural networks with many layers", metadata={"id":2}),
    Document(page_content="Statistics is foundational to ML", metadata={"id":3}),
    Document(page_content="Reinforcement learning deals with agents, rewards, and environments", metadata={"id":4}),
    Document(page_content="Optimization methods like gradient descent are used in training models", metadata={"id":5}),
]

# Build vector store
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vs = FAISS.from_documents(docs, embedding=emb)
base_retriever = vs.as_retriever(search_kwargs={"k":2})

# LLM for query generation
llm = ChatOllama(model="qwen2.5:latest", temperature=0.7)

# MultiQueryRetriever
multi_q = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True
)

# Query
query = "What is reinforcement learning?"
out_docs = multi_q.invoke(query)

print("MultiQuery Retriever Results:")
for d in out_docs:
    print(f"ID {d.metadata.get('id')}: {d.page_content}")

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What are the core principles and applications of reinforcement learning?', 'How does reinforcement learning work, and what are its main components?', 'Can you explain the concept and use cases of reinforcement learning in detail?']


MultiQuery Retriever Results:
ID 4: Reinforcement learning deals with agents, rewards, and environments
ID 1: Machine learning includes supervised, unsupervised, and reinforcement learning
